# AQI Dataset Validation

## Objectives:

This notebook validates the combine CPCB air poluution dataset before data cleaning and feature engineering.

#### Validation includes:

- Dataset loading
- Shape and memory inspection
- Column validation
- Data type validation
- Timestamp validation
- Geographic validation
- Pollutant availability
- Missing value 
- Dublicate detection
- Negative value detection 
- Extreme pollutant value detection
- Geographic coordinate validation
- Temporal coverage validation
- Station/location coverage
- Sampling frequency validation
- Final validation summary

The original dataset is not modified in this notebook.

# 1. Imort Libraries

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: f"{x:,.3f}")

# 2. Project Paths

In [3]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "Data"
RAW_DATA_DIR = DATA_DIR / "Raw"
REPORT_DIR = PROJECT_ROOT / "Reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
DATA_FILE = RAW_DATA_DIR / "AQI_Dataset.csv"

print("Project root :", PROJECT_ROOT.resolve())
print("Dataset path :", DATA_FILE.resolve())
print("Report path  :", REPORT_DIR.resolve())

Project root : /Users/syedfaizaanahmad/Desktop/Project/AI-Based-Enviro-Pollution-Forecasting-Sys-for-Persona-Health-Alerts---Smart-Route-Plann-Jun-2026
Dataset path : /Users/syedfaizaanahmad/Desktop/Project/AI-Based-Enviro-Pollution-Forecasting-Sys-for-Persona-Health-Alerts---Smart-Route-Plann-Jun-2026/Data/Raw/AQI_Dataset.csv
Report path  : /Users/syedfaizaanahmad/Desktop/Project/AI-Based-Enviro-Pollution-Forecasting-Sys-for-Persona-Health-Alerts---Smart-Route-Plann-Jun-2026/Reports


# 3. Check Dataset File

In [4]:
if not DATA_FILE.exists():
    raise FileNotFoundError (
        f"Dataset not found at : \n{DATA_FILE.resolve()}"
    )

file_size_mb = DATA_FILE.stat().st_size / (1024 ** 2)
file_size_gb = DATA_FILE.stat().st_size / (1024 ** 3)

print("Dataset found successfully.")
print(f"File size: {file_size_mb:,.2f} MB")
print(f"File size: {file_size_gb:,.2f} GB")

Dataset found successfully.
File size: 617.22 MB
File size: 0.60 GB


# 4. Read Dataset Header

In [5]:
sample_df = pd.read_csv(DATA_FILE, nrows=5)
print("Columns found in the dataset:")
print()

for i, column in enumerate(sample_df.columns, start = 1):
    print(f"{i:02d}.{column}")

Columns found in the dataset:

01.Timestamp
02.PM2.5
03.PM10
04.Nitric Oxide
05.Nitrogen Dioxide
06.Nitrogen Oxides
07.Ammonia
08.Sulfur Dioxide
09.Carbon Monoxide
10.Ozone
11.Ambient Temperature
12.Relative Humidity
13.Solar Radiation
14.Rainfall
15.State
16.City
17.Latitude
18.Longitude
19.Calculated_AQI
20.Month
21.Hour
22.DayOfWeek
23.Is_Weekend
24.State_Encoded
25.City_Encoded


In [ ]:
# 5. Display Raw Sample
display(sample_df)

,Timestamp,PM2.5,PM10,Nitric Oxide,Nitrogen Dioxide,Nitrogen Oxides,Ammonia,Sulfur Dioxide,Carbon Monoxide,Ozone,Ambient Temperature,Relative Humidity,Solar Radiation,Rainfall,State,City,Latitude,Longitude,Calculated_AQI,Month,Hour,DayOfWeek,Is_Weekend,State_Encoded,City_Encoded
0,2017-09-05 11:00:00,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500,33.800,69.000,372.000,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,11,1,0,0,3
1,2017-09-05 12:00:00,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500,33.800,69.000,372.000,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,12,1,0,0,3
2,2017-09-05 13:00:00,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500,33.800,69.000,372.000,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,13,1,0,0,3
3,2017-09-05 14:00:00,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500,33.800,69.000,372.000,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,14,1,0,0,3
4,2017-09-05 15:00:00,23.000,49.500,0.650,14.550,8.280,8.850,4.520,0.150,62.500,32.220,70.500,290.750,0.000,Andhra Pradesh,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987,81.736,90.106,9,15,1,0,0,3


# 6. Normalize column names for validation

In [7]:
original_columns = sample_df.columns.tolist()

normalized_columns = (sample_df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace(".", "", regex = False).str.replace("-", "_"))

column_mapping = dict(zip(original_columns, normalized_columns))
column_mapping

{'Timestamp': 'timestamp',
 'PM2.5': 'pm25',
 'PM10': 'pm10',
 'Nitric Oxide': 'nitric_oxide',
 'Nitrogen Dioxide': 'nitrogen_dioxide',
 'Nitrogen Oxides': 'nitrogen_oxides',
 'Ammonia': 'ammonia',
 'Sulfur Dioxide': 'sulfur_dioxide',
 'Carbon Monoxide': 'carbon_monoxide',
 'Ozone': 'ozone',
 'Ambient Temperature': 'ambient_temperature',
 'Relative Humidity': 'relative_humidity',
 'Solar Radiation': 'solar_radiation',
 'Rainfall': 'rainfall',
 'State': 'state',
 'City': 'city',
 'Latitude': 'latitude',
 'Longitude': 'longitude',
 'Calculated_AQI': 'calculated_aqi',
 'Month': 'month',
 'Hour': 'hour',
 'DayOfWeek': 'dayofweek',
 'Is_Weekend': 'is_weekend',
 'State_Encoded': 'state_encoded',
 'City_Encoded': 'city_encoded'}

# 7. Load complete dataset

In [8]:
print("Loading complete dataset...")

df = pd.read_csv(DATA_FILE, low_memory=False)
print("Dataset loaded successfully")
print(f"Rows : {len(df):,}")
print(f"Columns : {df.shape[1]:,}")

Loading complete dataset...
Dataset loaded successfully
Rows : 3,431,900
Columns : 25


# 8. Normalize DataFrame Columns Names

In [9]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "", regex=False)
    .str.replace("-", "_")
)

print("Normalized columns:")

for i, column in enumerate(df.columns, start=1):
    print(f"{i:02d}. {column}")

Normalized columns:
01. timestamp
02. pm25
03. pm10
04. nitric_oxide
05. nitrogen_dioxide
06. nitrogen_oxides
07. ammonia
08. sulfur_dioxide
09. carbon_monoxide
10. ozone
11. ambient_temperature
12. relative_humidity
13. solar_radiation
14. rainfall
15. state
16. city
17. latitude
18. longitude
19. calculated_aqi
20. month
21. hour
22. dayofweek
23. is_weekend
24. state_encoded
25. city_encoded


# 9. Dataset Shape

In [10]:
rows, columns = df.shape
print("=" * 50)
print("DATASET SHAPE")
print("=" * 50)

print(f"Total rows : {rows:,}")
print(f"Total columns : {columns:,}")

DATASET SHAPE
Total rows : 3,431,900
Total columns : 25


# 10. Dataset Memory Usage

In [11]:
memory_usage = df.memory_usage(deep=True)

total_memory_mb = memory_usage.sum() / (1024 ** 2)
total_memory_gb = memory_usage.sum() / (1024 ** 3)

print("=" * 50)
print("MEMORY USAGE")
print("=" * 50)

print(f"Total memory: {total_memory_mb:,.2f} MB")
print(f"Total memory: {total_memory_gb:,.2f} GB")

MEMORY USAGE
Total memory: 1,234.73 MB
Total memory: 1.21 GB


# 11. Memory Usage by column

In [12]:
memory_report = (
    df.memory_usage(deep=True)
    .sort_values(ascending=False)
    .reset_index()
)

memory_report.columns = ["Column", "Memory Usage (bytes)"]
memory_report["Memory Usage (MB)"] = memory_report["Memory Usage (bytes)"] / (1024 ** 2)

display(memory_report.head(20))

,Column,Memory Usage (bytes),Memory Usage (MB)
0,city,257467393,245.540
1,timestamp,233369200,222.558
2,state,199860684,190.602
3,solar_radiation,27455200,26.183
4,state_encoded,27455200,26.183
5,is_weekend,27455200,26.183
6,dayofweek,27455200,26.183
7,hour,27455200,26.183
8,month,27455200,26.183
9,calculated_aqi,27455200,26.183


# 12. Data Type Inspection

In [13]:
dtype_report = pd.DataFrame ({
    "Column": df.columns,
    "dtype": df.dtypes.astype(str).values
})

display(dtype_report)

,Column,dtype
0,timestamp,str
1,pm25,float64
2,pm10,float64
3,nitric_oxide,float64
4,nitrogen_dioxide,float64
5,nitrogen_oxides,float64
6,ammonia,float64
7,sulfur_dioxide,float64
8,carbon_monoxide,float64
9,ozone,float64


# 13. Expected Column Configuration

In [20]:
REQUIRED_POLLUTANTS = [
    "pm25",
    "pm10",
    "nitric_oxide",
    "nitrogen_dioxide",
    "nitrogen_oxides",
    "ammonia",
    "sulfur_dioxide",
    "carbon_monoxide",
    "ozone"
]

GEOGRAPHIC_COLUMNS = [
    "state",
    "city",
    "latitude",
    "longitude"
]

TIME_COLUMN_CANDIDATES = [
    "timestamp",
    "datetime",
    "date_time",
    "from_date",
    "date"
]

WEATHER_COLUMNS = [
    "temperature",
    "humidity",
    "wind_speed",
    "wind_direction",
    "pressure"
]

# 14. Detect Timestamp Column

In [21]:
timestamp_column = None

for column in TIME_COLUMN_CANDIDATES:
    if column in df.columns:
        timestamp_column = column
        break

    if timestamp_column:
        print(f"Timestamp column found: {timestamp_column}")

    else:
        print("WARNING: Timestamp column not detected")

# 15. Required Pollutant Validation

In [22]:
pollutant_validation = []
for pollutant in REQUIRED_POLLUTANTS:
    exists = pollutant in df.columns
    pollutant_validation.append({"pollutant": pollutant, "exists": exists})

pollutant_validation_df = pd.DataFrame(pollutant_validation)
display(pollutant_validation_df)

,pollutant,exists
0,pm25,True
1,pm10,True
2,nitric_oxide,True
3,nitrogen_dioxide,True
4,nitrogen_oxides,True
5,ammonia,True
6,sulfur_dioxide,True
7,carbon_monoxide,True
8,ozone,True


# 16. Missing Required Pollutants

In [23]:
available_pollutants = [
    col for col in REQUIRED_POLLUTANTS 
    if col in df.columns
]

missing_pollutants = [
    col for col in REQUIRED_POLLUTANTS 
    if col not in df.columns
]

print("Available pollutants:")
print(available_pollutants)
print()
print("Missing pollutants:")
print(missing_pollutants)

Available pollutants:
['pm25', 'pm10', 'nitric_oxide', 'nitrogen_dioxide', 'nitrogen_oxides', 'ammonia', 'sulfur_dioxide', 'carbon_monoxide', 'ozone']

Missing pollutants:
[]


# 17. Geographic column validation

In [24]:
geographic_validation = []
for column in GEOGRAPHIC_COLUMNS:
    geographic_validation.append({"column": column, "exists": column in df.columns})

geographic_validation_df = pd.DataFrame(geographic_validation)
display(geographic_validation_df)

,column,exists
0,state,True
1,city,True
2,latitude,True
3,longitude,True


# 18. Missing Value Analysis

In [26]:
missing_report = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isnull().sum().values
})

missing_report["missing_percentage"] = (
    missing_report["missing_count"] / len(df)
) * 100

missing_report = missing_report.sort_values(
    "missing_percentage",
    ascending=False
)

display(missing_report)

,column,missing_count,missing_percentage
0,timestamp,0,0.000
13,rainfall,0,0.000
23,state_encoded,0,0.000
22,is_weekend,0,0.000
21,dayofweek,0,0.000
20,hour,0,0.000
19,month,0,0.000
18,calculated_aqi,0,0.000
17,longitude,0,0.000
16,latitude,0,0.000


# 19. Save Missing value report

In [27]:
missing_report_path = (
    REPORT_DIR / "missing_value_report.csv"
)

missing_report.to_csv(
    missing_report_path,
    index=False
)

print(
    f"Missing value report saved to:\n"
    f"{missing_report_path.resolve()}"
)

Missing value report saved to:
/Users/syedfaizaanahmad/Desktop/Project/AI-Based-Enviro-Pollution-Forecasting-Sys-for-Persona-Health-Alerts---Smart-Route-Plann-Jun-2026/Reports/missing_value_report.csv


# 20. Pollutant Missing value analysis

In [28]:
pollutant_missing_report = []

for pollutant in available_pollutants:
    missing_count = df[pollutant].isnull().sum()
    missing_percentage = (missing_count / len(df)) * 100
    pollutant_missing_report.append({
        "pollutant": pollutant,
        "missing_count": missing_count,
        "missing_percentage": missing_percentage
    })

pollutant_missing_df = pd.DataFrame(pollutant_missing_report)
pollutant_missing_df = pollutant_missing_df.sort_values(
    "missing_percentage",
    ascending=False
)

display(pollutant_missing_df)

,pollutant,missing_count,missing_percentage
0,pm25,0,0.000
1,pm10,0,0.000
2,nitric_oxide,0,0.000
3,nitrogen_dioxide,0,0.000
4,nitrogen_oxides,0,0.000
5,ammonia,0,0.000
6,sulfur_dioxide,0,0.000
7,carbon_monoxide,0,0.000
8,ozone,0,0.000


# 21. Duplicate row detection

In [29]:
print("Checking duplicate rows...")

duplicate_count = df.duplicated().sum()

duplicate_percentage = (
    duplicate_count / len(df)
) * 100

print(f"Duplicate rows       : {duplicate_count:,}")
print(f"Duplicate percentage : {duplicate_percentage:.4f}%")

Checking duplicate rows...
Duplicate rows       : 21
Duplicate percentage : 0.0006%


# 22. Display duplicate samples

In [30]:
duplicate_samples = df[
    df.duplicated(keep=False)
].head(20)

if len(duplicate_samples) > 0:
    display(duplicate_samples)
else:
    print("No duplicate rows found")

,timestamp,pm25,pm10,nitric_oxide,nitrogen_dioxide,nitrogen_oxides,ammonia,sulfur_dioxide,carbon_monoxide,ozone,ambient_temperature,relative_humidity,solar_radiation,rainfall,state,city,latitude,longitude,calculated_aqi,month,hour,dayofweek,is_weekend,state_encoded,city_encoded
1085532,2026-03-05 15:00:00,53.500,106.000,6.000,23.500,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,15,3,0,7,20
1085533,2026-03-05 15:00:00,53.500,106.000,6.000,23.500,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,15,3,0,7,20
1085534,2026-03-05 16:00:00,49.000,95.500,6.000,38.500,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,16,3,0,7,20
1085535,2026-03-05 16:00:00,49.000,95.500,6.000,38.500,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,16,3,0,7,20
1085536,2026-03-05 17:00:00,61.500,114.250,6.000,27.000,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,17,3,0,7,20
1085537,2026-03-05 17:00:00,61.500,114.250,6.000,27.000,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,17,3,0,7,20
1085538,2026-03-05 18:00:00,94.000,175.250,6.000,93.500,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,18,3,0,7,20
1085539,2026-03-05 18:00:00,94.000,175.250,6.000,93.500,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,18,3,0,7,20
1085540,2026-03-05 19:00:00,133.750,254.000,6.000,46.250,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,19,3,0,7,20
1085541,2026-03-05 19:00:00,133.750,254.000,6.000,46.250,19.000,15.991,2.000,0.000,19.290,27.120,68.155,57.780,0.000,Gujarat,"GIDC, Nandesari (, )",22.407,73.093,90.106,3,19,3,0,7,20


# 23. Convert pollutant columns to numeric for validation

In [31]:
pollutant_numeric = pd.DataFrame(index = df.index)
for pollutant in available_pollutants:
    pollutant_numeric[pollutant] = pd.to_numeric(
        df[pollutant],
        errors="coerce"
    )

print("Pollutant validation completed. All pollutants converted to numeric types where possible.")
display(pollutant_numeric.head())

Pollutant validation completed. All pollutants converted to numeric types where possible.


,pm25,pm10,nitric_oxide,nitrogen_dioxide,nitrogen_oxides,ammonia,sulfur_dioxide,carbon_monoxide,ozone
0,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500
1,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500
2,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500
3,25.000,45.000,1.800,12.200,7.900,10.200,5.600,0.100,79.500
4,23.000,49.500,0.650,14.550,8.280,8.850,4.520,0.150,62.500


# 24. Detect Non-Numeric Pollutant values

In [32]:
non_numeric_report = []

for pollutant in available_pollutants:
    
    original_non_null = df[pollutant].notna()
    
    converted_null = pollutant_numeric[pollutant].isna()
    
    invalid_mask = (
        original_non_null &
        converted_null
    )
    
    invalid_count = invalid_mask.sum()
    
    non_numeric_report.append({
        "pollutant": pollutant,
        "non_numeric_count": invalid_count
    })

non_numeric_df = pd.DataFrame(
    non_numeric_report
)

display(non_numeric_df)

,pollutant,non_numeric_count
0,pm25,0
1,pm10,0
2,nitric_oxide,0
3,nitrogen_dioxide,0
4,nitrogen_oxides,0
5,ammonia,0
6,sulfur_dioxide,0
7,carbon_monoxide,0
8,ozone,0


# 25. Show Non-Numeric Samples

In [33]:
for pollutant in available_pollutants:
    numeric_series = pd.to_numeric(df[pollutant], errors="coerce")
    invalid_mask = df[pollutant].notna() & numeric_series.isna()
    invalid_values = df.loc[invalid_mask, pollutant].astype(str).value_counts().head(10)
    if len(invalid_values) > 0:
        print("=" * 50)
        print(f"INVALID VALUES: {pollutant}")
        print("=" * 50)

        display(invalid_values)

# 26. Negative Pollutant value detection

In [37]:
negative_report = []
for pollutant in available_pollutants:
    negative_counts = (
        pollutant_numeric[pollutant] < 0
    ).sum()

    negative_percentage = (negative_counts / len(df)) * 100
    negative_report.append ({
        "pollutant": pollutant,
        "negative_count": negative_counts,
        "negative_percentage": negative_percentage
    })

negative_df = pd.DataFrame (negative_report)
display(negative_df)

,pollutant,negative_count,negative_percentage
0,pm25,0,0.000
1,pm10,0,0.000
2,nitric_oxide,0,0.000
3,nitrogen_dioxide,0,0.000
4,nitrogen_oxides,0,0.000
5,ammonia,0,0.000
6,sulfur_dioxide,0,0.000
7,carbon_monoxide,0,0.000
8,ozone,0,0.000


# 27. Pollutant Stattistical Summary

In [38]:
pollutant_summary = (
    pollutant_numeric[available_pollutants].describe().T
)

pollutant_summary["missing_count"] = (pollutant_numeric[available_pollutants].isnull().sum())
display(pollutant_summary)

,count,mean,std,min,25%,50%,75%,max,missing_count
pm25,"3,431,900.000",53.904,60.334,0.000,21.000,36.165,64.250,999.990,0
pm10,"3,431,900.000",114.703,106.477,0.000,48.710,82.750,142.020,"1,000.000",0
nitric_oxide,"3,431,900.000",13.617,30.173,0.000,2.780,6.000,12.320,500.000,0
nitrogen_dioxide,"3,431,900.000",24.433,27.314,0.000,8.860,16.380,29.168,499.990,0
nitrogen_oxides,"3,431,900.000",28.199,36.770,0.000,10.400,19.000,32.640,500.000,0
ammonia,"3,431,900.000",21.164,22.199,0.000,8.500,15.991,25.620,499.990,0
sulfur_dioxide,"3,431,900.000",13.050,15.724,0.000,4.835,8.820,15.500,200.000,0
carbon_monoxide,"3,431,900.000",0.832,0.841,0.000,0.360,0.640,1.013,41.670,0
ozone,"3,431,900.000",27.632,27.449,0.000,9.600,19.290,35.900,932.000,0


# 28. Define Extreme value limits

In [40]:
EXTREME_LIMITS = {
    "pm25": 1000,
    "pm10": 2000,
    "nitric_oxide": 1000,
    "nitrogen_dioxide": 1000,
    "nitrogen_oxides": 2000,
    "ammonia": 2000,
    "sulfur_dioxide	": 2000,
    "carbon_monoxide": 100,
    "ozone": 1000
}

# 29. Extreme Pollutant Value Detection

In [41]:
extreme_value_report = []
for pollutant in available_pollutants:
    if pollutant not in EXTREME_LIMITS:
        continue
    limit = EXTREME_LIMITS[pollutant]
    extreme_count = (pollutant_numeric[pollutant] > limit).sum()
    extreme_percentage = (extreme_count / len(df)) * 100
    
    extreme_value_report.append({
        "pollutant": pollutant,
        "validation_limit": limit,
        "extreme_count": extreme_count,
        "extreme_percentage": extreme_percentage
    })

extreme_value_df = pd.DataFrame(extreme_value_report)
display(extreme_value_df)

,pollutant,validation_limit,extreme_count,extreme_percentage
0,pm25,1000,0,0.000
1,pm10,2000,0,0.000
2,nitric_oxide,1000,0,0.000
3,nitrogen_dioxide,1000,0,0.000
4,nitrogen_oxides,2000,0,0.000
5,ammonia,2000,0,0.000
6,carbon_monoxide,100,0,0.000
7,ozone,1000,0,0.000


# 30. Show Extreme Value Samples

In [42]:
for pollutant in available_pollutants:
    if pollutant not in EXTREME_LIMITS:
        continue
    
    limit = EXTREME_LIMITS[pollutant]
    extreme_mask = (pollutant_numeric[pollutant] > limit)
    extreme_samples = df.loc[extreme_mask].head(5)
    
    if len(extreme_samples) > 0:
        print("=" * 60)
        print(
            f"EXTREME VALUES: {pollutant.upper()} "
            f"> {limit}"
        )
        print("=" * 60)
        
        display(extreme_samples)

# 31. Timestamp Validation

In [44]:
if timestamp_column:
    
    parsed_timestamp = pd.to_datetime(df[timestamp_column],errors="coerce")
    invalid_timestamp_count = (parsed_timestamp.isna().sum())
    invalid_timestamp_percentage = (invalid_timestamp_count / len(df)) * 100
    
    print(
        f"Invalid timestamps: "
        f"{invalid_timestamp_count:,}"
    )
    
    print(
        f"Invalid percentage: "
        f"{invalid_timestamp_percentage:.4f}%"
    )

Invalid timestamps: 0
Invalid percentage: 0.0000%


# 32. Temporal Coverage

In [45]:
if timestamp_column:
    valid_timestamps = parsed_timestamp.dropna()
    print("=" * 50)
    print("TEMPORAL COVERAGE")
    print("=" * 50)

    print("Start date:",valid_timestamps.min())
    print("End date:",valid_timestamps.max())
    print(
        "Total duration:",
        valid_timestamps.max()
        - valid_timestamps.min()
    )

TEMPORAL COVERAGE
Start date: 2017-01-01 00:00:00
End date: 2026-06-30 23:00:00
Total duration: 3467 days 23:00:00


# 33. Year-wise Row Distribution

In [46]:
if timestamp_column:
    
    year_distribution = (
        parsed_timestamp
        .dt.year
        .value_counts()
        .sort_index()
        .rename_axis("year")
        .reset_index(name="row_count")
    )
    
    display(year_distribution)

,year,row_count
0,2017,44163
1,2018,139613
2,2019,247019
3,2020,357092
4,2021,398331
5,2022,476284
6,2023,534440
7,2024,520894
8,2025,494319
9,2026,219745


# 34. Check 2026 Data

In [48]:
if timestamp_column:
    
    rows_2026 = (parsed_timestamp.dt.year == 2026).sum()

    print(f"2026 rows found: {rows_2026:,}")
    
    if rows_2026 > 0:
        print("2026 data is available")
    else:
        print("WARNING: No 2026 data detected")

2026 rows found: 219,745
2026 data is available


# 35. Month Wise Converage

In [47]:
if timestamp_column:
    
    month_distribution = (
        parsed_timestamp
        .dt.to_period("M")
        .value_counts()
        .sort_index()
        .rename_axis("year_month")
        .reset_index(name="row_count")
    )
    
    display(month_distribution.tail(30))

,year_month,row_count
84,2024-01,44346
85,2024-02,42371
86,2024-03,44516
87,2024-04,42640
88,2024-05,43342
89,2024-06,41121
90,2024-07,44501
91,2024-08,43278
92,2024-09,42323
93,2024-10,44436


# 36. State Validation

In [51]:
if "state" in df.columns:
    
    state_count = df["state"].nunique(
        dropna=True
    )
    
    print(f"Unique states: {state_count}")
    
    state_distribution = (
        df["state"]
        .value_counts(dropna=False)
        .rename_axis("state")
        .reset_index(name="row_count")
    )
    
    display(state_distribution)

Unique states: 19


,state,row_count
0,Delhi,355486
1,Gujarat,282609
2,Karnataka,273727
3,Harayana,267875
4,West Bengal,262816
5,Madhya Pradesh,261154
6,Andhra Pradesh,256378
7,Maharashtra,229190
8,Punjab,221882
9,Uttar Pradesh,221581


# 37. City Validation

In [52]:
if "city" in df.columns:
    
    city_count = df["city"].nunique(dropna=True)
    print(f"Unique cities: {city_count}")
    city_distribution = (
        df["city"]
        .value_counts(dropna=False)
        .rename_axis("city")
        .reset_index(name="row_count")
    )
    
    display(city_distribution.head(50))

Unique cities: 66


,city,row_count
0,"Golden Temple, Amritsar (, )",79744
1,"Maninagar, Ahmedabad ( , )",76922
2,"GVM Corporation, Visakhapatnam (, )",76310
3,"Anand Kala Kshetram, Rajamahendravaram (, )",75833
4,"Anand Vihar ( , )",75769
5,"Aya Nagar (, )",75319
6,"Bollaram Industrial Area, Hyderabad (, )",74989
7,"MIDC Khutala, Chandrapur (, )",73781
8,"Civil Line, Jalandhar (, )",73619
9,"More Chowk Waluj, Aurangabad (, )",73152


# 38. Sate and City Coverage

In [53]:
if (
    "state" in df.columns and
    "city" in df.columns
):
    
    state_city_coverage = (
        df.groupby("state")["city"]
        .nunique()
        .sort_values(ascending=False)
        .rename("city_count")
        .reset_index()
    )
    
    display(state_city_coverage)

,state,city_count
0,Karnataka,5
1,Uttar Pradesh,5
2,Bihar,5
3,Madhya Pradesh,5
4,Delhi,5
5,Gujarat,5
6,Harayana,5
7,Kerala,5
8,Andhra Pradesh,4
9,Maharashtra,4


# 39. Latitude Validation

In [54]:
if "latitude" in df.columns:
    latitude_numeric = pd.to_numeric(df["latitude"],errors="coerce")
    
    invalid_latitude = ((latitude_numeric < -90) | (latitude_numeric > 90))
    
    print("Invalid latitude values:", invalid_latitude.sum())
    
    print(
        "Latitude range:",
        latitude_numeric.min(),
        "to",
        latitude_numeric.max()
    )

Invalid latitude values: 0
Latitude range: 8.8787 to 31.620132


# 40. Longitude Validation

In [55]:
if "longitude" in df.columns:
    
    longitude_numeric = pd.to_numeric(df["longitude"], errors="coerce")
    invalid_longitude = ((longitude_numeric < -180) |(longitude_numeric > 180))
    
    print("Invalid longitude values:", invalid_longitude.sum())
    print(
        "Longitude range:",
        longitude_numeric.min(),
        "to",
        longitude_numeric.max()
    )

Invalid longitude values: 0
Longitude range: 72.5983 to 94.6404


# 41. India Coordiante Range Validation

In [56]:
if (
    "latitude" in df.columns and
    "longitude" in df.columns
):
    
    india_coordinate_mask = (
        latitude_numeric.between(6, 38) &
        longitude_numeric.between(68, 98)
    )
    
    outside_india_count = (
        ~india_coordinate_mask
    ).sum()
    
    outside_india_percentage = (
        outside_india_count / len(df)
    ) * 100
    
    print(
        f"Rows outside India coordinate range: "
        f"{outside_india_count:,}"
    )
    
    print(
        f"Percentage: "
        f"{outside_india_percentage:.4f}%"
    )

Rows outside India coordinate range: 0
Percentage: 0.0000%


# 42. Unique Monitoring Locations

In [57]:
if (
    "latitude" in df.columns and
    "longitude" in df.columns
):
    
    unique_locations = (
        df[
            ["latitude", "longitude"]
        ]
        .drop_duplicates()
    )
    
    print(
        "Unique monitoring locations:",
        f"{len(unique_locations):,}"
    )
    
    display(unique_locations.head(20))

Unique monitoring locations: 66


,latitude,longitude
0,16.987,81.736
75833,17.720,83.300
152143,14.675,77.597
184677,16.515,80.518
256378,27.103,93.701
288502,26.989,94.640
321142,26.192,91.695
352674,25.860,85.779
388172,24.757,84.366
424180,26.805,84.506


# 43. State - Location Coverage

In [58]:
if all(
    col in df.columns
    for col in [
        "state",
        "latitude",
        "longitude"
    ]
):
    
    state_location_coverage = (
        df.groupby("state")
        .apply(
            lambda x: x[
                ["latitude", "longitude"]
            ]
            .drop_duplicates()
            .shape[0]
        )
        .rename("location_count")
        .sort_values(ascending=False)
        .reset_index()
    )
    
    display(state_location_coverage)

,state,location_count
0,Karnataka,5
1,Uttar Pradesh,5
2,Bihar,5
3,Madhya Pradesh,5
4,Delhi,5
5,Gujarat,5
6,Harayana,5
7,Kerala,5
8,Andhra Pradesh,4
9,Maharashtra,4


# 44. Timestamp Dublicate Validation Per Location

In [59]:
location_time_columns = []

if "latitude" in df.columns:
    location_time_columns.append("latitude")

if "longitude" in df.columns:
    location_time_columns.append("longitude")

if timestamp_column:
    location_time_columns.append(timestamp_column)

if len(location_time_columns) >= 3:
    
    duplicate_location_time = (
        df.duplicated(
            subset=location_time_columns
        )
        .sum()
    )
    
    print(
        "Duplicate location-timestamp rows:",
        f"{duplicate_location_time:,}"
    )

Duplicate location-timestamp rows: 2,780


# 45. Sampling Frequency Analysis

In [60]:
if (timestamp_column and "latitude" in df.columns and "longitude" in df.columns):
    
    sampling_df = pd.DataFrame({
        "latitude": df["latitude"],
        "longitude": df["longitude"],
        "timestamp": parsed_timestamp})
    
    sampling_df = sampling_df.dropna()
    
    sampling_df = sampling_df.sort_values(
        [
            "latitude",
            "longitude",
            "timestamp"
        ]
    )
    
    sampling_df["time_difference"] = (
        sampling_df
        .groupby(
            ["latitude", "longitude"]
        )["timestamp"]
        .diff()
    )
    
    sampling_frequency = (
        sampling_df["time_difference"]
        .value_counts()
        .head(20)
        .rename_axis("time_difference")
        .reset_index(name="count")
    )
    
    display(sampling_frequency)

,time_difference,count
0,0 days 01:00:00,3420211
1,0 days 00:00:00,2780
2,0 days 02:00:00,878
3,0 days 05:00:00,747
4,0 days 03:00:00,554
5,0 days 04:00:00,467
6,0 days 06:00:00,363
7,0 days 08:00:00,344
8,0 days 10:00:00,329
9,0 days 11:00:00,323


# 46. Check Large Time Gaps

In [61]:
if "sampling_df" in globals():
    
    large_gap_threshold = pd.Timedelta(
        hours=24
    )
    
    large_gaps = (
        sampling_df["time_difference"]
        > large_gap_threshold
    )
    
    large_gap_count = large_gaps.sum()
    
    print(
        "Time gaps greater than 24 hours:",
        f"{large_gap_count:,}"
    )

Time gaps greater than 24 hours: 1,891


# 47. Pollutant Availability by State

In [62]:
if "state" in df.columns:
    
    pollutant_state_availability = []
    
    for state, state_df in df.groupby("state"):
        
        record = {
            "state": state,
            "total_rows": len(state_df)
        }
        
        for pollutant in available_pollutants:
            
            availability = (state_df[pollutant].notna().mean() * 100)
            record[f"{pollutant}_availability"] = availability
        pollutant_state_availability.append(record)
    
    pollutant_state_df = pd.DataFrame(pollutant_state_availability)
    display(pollutant_state_df)

,state,total_rows,pm25_availability,pm10_availability,nitric_oxide_availability,nitrogen_dioxide_availability,nitrogen_oxides_availability,ammonia_availability,sulfur_dioxide_availability,carbon_monoxide_availability,ozone_availability
0,Andhra Pradesh,256378,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
1,Arunachal Pradesh,32124,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
2,Assam,64172,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
3,Bihar,186427,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
4,Chandigarh,41796,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
5,Chhattisgarh,111785,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
6,Delhi,355486,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
7,Gujarat,282609,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
8,Harayana,267875,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
9,Himachal Pradesh,37157,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000


# 48. Pollutant Availability by Year

In [63]:
if timestamp_column:
    
    pollutant_year_df = (pollutant_numeric.copy())
    pollutant_year_df["year"] = (parsed_timestamp.dt.year)
    
    year_availability = (
        pollutant_year_df
        .groupby("year")[
            available_pollutants
        ]
        .apply(
            lambda x: x.notna().mean() * 100
        )
    )
    
    display(year_availability)

,pm25,pm10,nitric_oxide,nitrogen_dioxide,nitrogen_oxides,ammonia,sulfur_dioxide,carbon_monoxide,ozone
year,,,,,,,,,
2017,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
2018,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
2019,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
2020,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
2021,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
2022,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
2023,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
2024,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
2025,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000


# 49. Rows with all pollutants missing

In [64]:
all_pollutants_missing = (
    pollutant_numeric[
        available_pollutants
    ]
    .isnull()
    .all(axis=1)
)

all_missing_count = (
    all_pollutants_missing.sum()
)

all_missing_percentage = (
    all_missing_count / len(df)
) * 100

print(
    "Rows with all pollutants missing:",
    f"{all_missing_count:,}"
)

print(
    "Percentage:",
    f"{all_missing_percentage:.4f}%"
)

Rows with all pollutants missing: 0
Percentage: 0.0000%


# 50. Rows with partial pollutant data

In [65]:
pollutant_non_null_count = (
    pollutant_numeric[
        available_pollutants
    ]
    .notna()
    .sum(axis=1)
)

pollutant_completeness_distribution = (
    pollutant_non_null_count
    .value_counts()
    .sort_index()
    .rename_axis(
        "available_pollutant_count"
    )
    .reset_index(name="row_count")
)

display(
    pollutant_completeness_distribution
)

,available_pollutant_count,row_count
0,9,3431900


# 51. Dataset Validation Summary 

In [66]:
validation_summary = {
    "total_rows": len(df),
    "total_columns": df.shape[1],
    "memory_gb": total_memory_gb,
    "duplicate_rows": duplicate_count,
    "available_pollutants": len(
        available_pollutants
    ),
    "missing_pollutants": len(
        missing_pollutants
    ),
    "all_pollutants_missing_rows": (
        all_missing_count
    )
}

if timestamp_column:
    validation_summary[
        "invalid_timestamps"
    ] = invalid_timestamp_count
    
    validation_summary[
        "start_date"
    ] = valid_timestamps.min()
    
    validation_summary[
        "end_date"
    ] = valid_timestamps.max()

if "state" in df.columns:
    validation_summary[
        "unique_states"
    ] = df["state"].nunique()

if "city" in df.columns:
    validation_summary[
        "unique_cities"
    ] = df["city"].nunique()

if (
    "latitude" in df.columns and
    "longitude" in df.columns
):
    validation_summary[
        "unique_locations"
    ] = len(unique_locations)

validation_summary_df = pd.DataFrame(
    validation_summary.items(),
    columns=[
        "validation_metric",
        "value"
    ]
)

display(validation_summary_df)

,validation_metric,value
0,total_rows,3431900
1,total_columns,25
2,memory_gb,1.206
3,duplicate_rows,21
4,available_pollutants,9
5,missing_pollutants,0
6,all_pollutants_missing_rows,0
7,invalid_timestamps,0
8,start_date,2017-01-01 00:00:00
9,end_date,2026-06-30 23:00:00


# 52. Save validation Summary

In [67]:
validation_summary_path = (
    REPORT_DIR /
    "data_validation_summary.csv"
)

validation_summary_df.to_csv(
    validation_summary_path,
    index=False
)

print(
    "Validation summary saved to:"
)

print(
    validation_summary_path.resolve()
)

Validation summary saved to:
/Users/syedfaizaanahmad/Desktop/Project/AI-Based-Enviro-Pollution-Forecasting-Sys-for-Persona-Health-Alerts---Smart-Route-Plann-Jun-2026/Reports/data_validation_summary.csv


# 53. Save Pollutant Validation Report

In [68]:
pollutant_missing_df.to_csv(
    REPORT_DIR /
    "pollutant_missing_report.csv",
    index=False
)

negative_df.to_csv(
    REPORT_DIR /
    "negative_pollutant_report.csv",
    index=False
)

extreme_value_df.to_csv(
    REPORT_DIR /
    "extreme_pollutant_report.csv",
    index=False
)

non_numeric_df.to_csv(
    REPORT_DIR /
    "non_numeric_pollutant_report.csv",
    index=False
)

print(
    "Pollutant validation reports saved"
)

Pollutant validation reports saved


# Final Validation Status

In [69]:
critical_issues = []

if missing_pollutants:
    critical_issues.append(
        f"{len(missing_pollutants)} "
        f"pollutant columns missing"
    )

if timestamp_column is None:
    critical_issues.append(
        "Timestamp column missing"
    )

if duplicate_count > 0:
    critical_issues.append(
        f"{duplicate_count:,} "
        f"duplicate rows detected"
    )

if all_missing_count > 0:
    critical_issues.append(
        f"{all_missing_count:,} rows "
        f"have all pollutants missing"
    )

print("=" * 60)
print("FINAL DATA VALIDATION STATUS")
print("=" * 60)

if len(critical_issues) == 0:
    
    print(
        "STATUS: DATASET PASSED BASIC VALIDATION"
    )
    
    print(
        "Dataset is ready for data cleaning."
    )

else:
    
    print(
        "STATUS: VALIDATION ISSUES DETECTED"
    )
    
    print()
    
    for i, issue in enumerate(
        critical_issues,
        start=1
    ):
        print(f"{i}. {issue}")

print("=" * 60)

FINAL DATA VALIDATION STATUS
STATUS: VALIDATION ISSUES DETECTED

1. 21 duplicate rows detected
